<img src="https://www.openquantum.com/OpenQuantumByQuantumRings.png" alt="Open Quantum" width="150" align="right"/>

# Getting Started with Open Quantum
### Presented by Quantum Rings
Run quantum circuits on real quantum computers in seconds. This one notebook works everywhere: in your browser on [openquantum.com/notebooks](https://www.openquantum.com/notebooks), in Google Colab, or locally with Jupyter.

Sign up for free on the [Open Quantum Website](https://www.openquantum.com/signup) to get started!


## 1. Set up the environment
Installs the Open Quantum SDK. The `pyodide-http` patch only does anything when running in a browser; everywhere else it is a no-op.


In [ ]:
%pip install -q openquantum-sdk pyodide-http matplotlib
import pyodide_http
pyodide_http.patch_all()
print('Environment ready.')


## 2. Connect to Open Quantum
The SDK authenticates with an SDK key everywhere it runs: here, in Colab, or on your own machine. [Create a key in your account settings](https://www.openquantum.com/settings/keys), then paste the client ID and secret below. The secret is shown only once at creation; if you lost it, delete the key and create a new one.

In [ ]:
CLIENT_ID = ''      # paste your SDK key credentials here
CLIENT_SECRET = ''

from openquantum_sdk.clients import SchedulerClient, ManagementClient

scheduler = management = None
if CLIENT_ID and CLIENT_SECRET:
    from openquantum_sdk.auth import ClientCredentials, ClientCredentialsAuth
    auth = ClientCredentialsAuth(creds=ClientCredentials(
        client_id=CLIENT_ID, client_secret=CLIENT_SECRET))
    scheduler = SchedulerClient(auth=auth)
    management = ManagementClient(auth=auth)
    print('Authenticated with your SDK key.')
else:
    print('Not connected. Paste your SDK key above and re-run this cell.')

## 3. Run a Quantum Circuit on Open Quantum
Choose from several QPUs from leading providers like IonQ, Rigetti, and IQM. Submit your OpenQASM circuit. Receive results in minutes!


In [ ]:
# Fetch Online QPUs
classes = management.list_backend_classes()
available_qpus = [bc.short_code for bc in classes.backend_classes if bc.accepting_jobs]
print('Available QPUs')
print(available_qpus)

# Select Execution Parameters
BACKEND_CLASS = available_qpus[0]  # Select a QPU with its shortcode.
JOB_CATEGORY = 'phys:hds'  # The category that best describes your experiment.
NAME = 'Open Quantum Demo'  # Name your jobs to keep better track of them!
SHOTS = 100  # Integer number of shots for your experiment.

# Insert your QASM circuit here
MY_QASM = b'''
OPENQASM 3.0;
include "stdgates.inc";
bit[3] meas;
qubit[3] q;
h q[0];
cx q[0], q[1];
cx q[0], q[2];
barrier q[0], q[1], q[2];
meas[0] = measure q[0];
meas[1] = measure q[1];
meas[2] = measure q[2];
'''


In [ ]:
import json
from openquantum_sdk.clients import JobSubmissionConfig
from openquantum_sdk.enums import ExecutionPlanType, QueuePriorityType

CONFIG = JobSubmissionConfig(
    backend_class_id=BACKEND_CLASS,
    name=NAME,
    job_subcategory_id=JOB_CATEGORY,
    shots=SHOTS,
    execution_plan=ExecutionPlanType.PUBLIC,
    queue_priority=QueuePriorityType.STANDARD,
    auto_approve_quote=True,
    verbose=True,
)

job = scheduler.submit_job(CONFIG, file_content=MY_QASM)

print('\n--- JOB SUCCEEDED ---')
print(f'Job ID: {job.id}')
print(f'Status: {job.status}')

output_json = None
if job.output_data_url:
    output_json = scheduler.download_job_output(job)
    print('\n--- JOB OUTPUT (JSON) ---')
    print(json.dumps(output_json, indent=2))
else:
    print('No output available.')


In [ ]:
# Plot the measurement counts (plain matplotlib, works everywhere)
import matplotlib.pyplot as plt

if isinstance(output_json, dict) and output_json:
    plt.figure(figsize=(6, 4))
    plt.bar(list(output_json.keys()), list(output_json.values()), color='#50bc83')
    plt.title('Job Results')
    plt.xlabel('Outcome')
    plt.ylabel('Counts')
    plt.show()


## What next
- Swap `MY_QASM` for your own OpenQASM circuit and re-run.
- [All notebooks](https://www.openquantum.com/notebooks), including PennyLane and qBraid.
- [Continue your Getting Started guide](https://www.openquantum.com/getting-started).
